# NeuralOps A100 Inference Server

Runs **Llama 3.1 70B** on your A100 via vLLM, exposes it as an OpenAI-compatible API via ngrok, and registers it as a live provider in NeuralOps.

**Runtime:** A100 GPU (Runtime > Change runtime type > A100)

**What this does:**
1. Installs vLLM (fastest open-source inference engine)
2. Downloads Llama 3.1 70B in 4-bit quantization (fits in A100 VRAM)
3. Starts an OpenAI-compatible server on port 8000
4. Exposes it publicly via ngrok tunnel
5. Tests the endpoint with a sample prompt
6. Shows how to plug it into NeuralOps router

**Cost:** Free on Colab Pro+ A100. Uses ~35GB VRAM with 4-bit quantization.

In [1]:
# Cell 1: Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Mon Aug 10 04:34:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Cell 2: Install dependencies
# vLLM is the fastest open-source LLM inference engine
# It handles continuous batching, PagedAttention, and tensor parallelism
!pip install vllm==0.5.5 --quiet
!pip install pyngrok --quiet
!pip install httpx --quiet
print('Dependencies installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.6/134.6 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 133.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 103.0 MB/s eta

In [3]:
# Cell 3: Configure ngrok
# Get your free token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = '3Hi1gba2TqaGG9DMWgEZ2BlyU1x_7xCgMWSHEuWs5exMa1JtT'  # paste your ngrok token here

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
print('ngrok configured.')

ngrok configured.


In [19]:
import subprocess
import time
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Use Llama 3.1 8B instead -- fits easily, still impressive
MODEL = 'hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4'

print(f'Starting vLLM server with {MODEL}')
print('Estimated startup time: 2-3 minutes')

server_process = subprocess.Popen(
    [
        'python', '-m', 'vllm.entrypoints.openai.api_server',
        '--model', MODEL,
        '--host', '0.0.0.0',
        '--port', '8000',
        '--max-model-len', '8192',
        '--gpu-memory-utilization', '0.90',
        '--enforce-eager',
        '--dtype', 'auto',
        '--served-model-name', 'llama-3.1-8b',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

print(f'Server process started (PID {server_process.pid})')
print('Waiting for server to be ready...')

Starting vLLM server with hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4
Estimated startup time: 2-3 minutes
Server process started (PID 7921)
Waiting for server to be ready...


In [20]:
# Cell 5: Wait for server ready + stream logs
import httpx
import time
import threading

def stream_logs():
    for line in server_process.stdout:
        print(f'[vLLM] {line}', end='')

log_thread = threading.Thread(target=stream_logs, daemon=True)
log_thread.start()

# Poll health endpoint until ready
max_wait = 600  # 10 minutes
start = time.time()
ready = False

while time.time() - start < max_wait:
    try:
        resp = httpx.get('http://localhost:8000/health', timeout=3)
        if resp.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)

if ready:
    elapsed = time.time() - start
    print(f'\nvLLM server ready in {elapsed:.0f}s')
else:
    print('Server did not start in time. Check logs above.')

[vLLM] (APIServer pid=7921) INFO 08-10 04:53:38 [api_utils.py:345] 
[vLLM] (APIServer pid=7921) INFO 08-10 04:53:38 [api_utils.py:345]        █     █     █▄   ▄█
[vLLM] (APIServer pid=7921) INFO 08-10 04:53:38 [api_utils.py:345]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.26.0
[vLLM] (APIServer pid=7921) INFO 08-10 04:53:38 [api_utils.py:345]   █▄█▀ █     █     █     █  model   hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4
[vLLM] (APIServer pid=7921) INFO 08-10 04:53:38 [api_utils.py:345]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
[vLLM] (APIServer pid=7921) INFO 08-10 04:53:38 [api_utils.py:345] 
[vLLM] (APIServer pid=7921) INFO 08-10 04:53:38 [api_utils.py:273] non-default args: {'host': '0.0.0.0', 'model': 'hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4', 'max_model_len': 8192, 'enforce_eager': True, 'served_model_name': ['llama-3.1-8b'], 'gpu_memory_utilization': 0.9}
[vLLM] (APIServer pid=7921) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable hig

In [12]:
!pip uninstall torchaudio -y
print('torchaudio removed.')

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
torchaudio removed.


In [9]:
!pip install torchaudio --upgrade --quiet --index-url https://download.pytorch.org/whl/cu126
print('torchaudio fixed.')

torchaudio fixed.


In [6]:
!pip install vllm --upgrade --quiet
!pip install torchaudio --upgrade --quiet
print('vLLM upgraded.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.7/303.7 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45

In [21]:
# Cell 6: Expose via ngrok
from pyngrok import ngrok

tunnel = ngrok.connect(8000, 'http')
public_url = tunnel.public_url

print('=' * 60)
print(f'Public URL: {public_url}')
print(f'API base:   {public_url}/v1')
print(f'Model:      llama-3.1-70b')
print('=' * 60)
print()
print('Add to your NeuralOps router.py:')
print(f'''
ProviderConfig(
    name=Provider.LOCAL_A100,
    base_url="{public_url}/v1/chat/completions",
    api_key_env="LOCAL_API_KEY",
    model="llama-3.1-70b",
),
''')

Public URL: https://blazing-twister-lustrous.ngrok-free.dev
API base:   https://blazing-twister-lustrous.ngrok-free.dev/v1
Model:      llama-3.1-70b

Add to your NeuralOps router.py:

ProviderConfig(
    name=Provider.LOCAL_A100,
    base_url="https://blazing-twister-lustrous.ngrok-free.dev/v1/chat/completions",
    api_key_env="LOCAL_API_KEY",
    model="llama-3.1-70b",
),



In [23]:
# Cell 7: Test the endpoint
import httpx
import time

prompts = [
    'Explain causal inference in one sentence.',
    'What is the CAP theorem?',
    'Why does observability matter for AI agents?',
]

print('Testing local Llama 3.1 8B endpoint...')
print()

for prompt in prompts:
    t0 = time.perf_counter()
    resp = httpx.post(
        'http://localhost:8000/v1/chat/completions',
        json={
            'model': 'llama-3.1-8b',
            'messages': [{'role': 'user', 'content': prompt}],
            'max_tokens': 128,
            'temperature': 0.7,
        },
        timeout=60.0,
    )
    latency = (time.perf_counter() - t0) * 1000
    data = resp.json()
    content = data['choices'][0]['message']['content']
    tokens = data['usage']['completion_tokens']
    tps = tokens / (latency / 1000)

    print(f'Prompt:  {prompt}')
    print(f'Answer:  {content[:200]}')
    print(f'Latency: {latency:.0f}ms | Tokens: {tokens} | Speed: {tps:.0f} tok/s')
    print()

Testing local Llama 3.1 8B endpoint...

[vLLM] (APIServer pid=7921) INFO:     127.0.0.1:54418 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Prompt:  Explain causal inference in one sentence.
Answer:  Causal inference is the process of drawing conclusions about cause-and-effect relationships between variables, typically involving the use of statistical models, experiments, and observational data to
Latency: 1417ms | Tokens: 42 | Speed: 30 tok/s

[vLLM] (APIServer pid=7921) INFO 08-10 04:58:03 [loggers.py:310] Engine 000: Avg prompt throughput: 6.8 tokens/s, Avg generation throughput: 13.6 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 19.0%
[vLLM] (APIServer pid=7921) INFO:     127.0.0.1:41602 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Prompt:  What is the CAP theorem?
Answer:  The CAP theorem, also known as the Brewer's CAP theorem, states that it is impossible for a distributed data store to simultaneously guarantee all three of the fol

In [25]:
# Cell 8: Benchmark against free API providers
# Compare your local A100 Llama 70B against Groq and Mistral

import httpx
import asyncio
import time
import os

GROQ_KEY    = ''  # set via environment variable
MISTRAL_KEY = ''  # set via environment variable

PROMPT = 'What are the three laws of thermodynamics? Be concise.'

async def call_provider(name, url, model, api_key, extra_headers=None):
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json',
        **(extra_headers or {}),
    }
    payload = {
        'model': model,
        'messages': [{'role': 'user', 'content': PROMPT}],
        'max_tokens': 200,
    }
    t0 = time.perf_counter()
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(url, json=payload, headers=headers)
    latency = (time.perf_counter() - t0) * 1000
    data = resp.json()
    content = data['choices'][0]['message']['content']
    tokens = data.get('usage', {}).get('completion_tokens', 0)
    return name, content, latency, tokens

async def run_comparison():
    tasks = [
        call_provider(
        'Local A100 (Llama 3.1 8B)',
        'http://localhost:8000/v1/chat/completions',
        'llama-3.1-8b',
        'local',
),
    ]
    if GROQ_KEY:
        tasks.append(call_provider(
            'Groq (Llama 3.3 70B)',
            'https://api.groq.com/openai/v1/chat/completions',
            'llama-3.3-70b-versatile',
            GROQ_KEY,
        ))
    if MISTRAL_KEY:
        tasks.append(call_provider(
            'Mistral Small',
            'https://api.mistral.ai/v1/chat/completions',
            'mistral-small-latest',
            MISTRAL_KEY,
        ))

    results = await asyncio.gather(*tasks)

    print(f'Prompt: {PROMPT}')
    print()
    print(f'{"Provider":<35} {"Latency":>10} {"Tokens":>8} {"Speed":>12}')
    print('-' * 70)
    for name, content, latency, tokens in results:
        tps = tokens / (latency / 1000) if latency > 0 else 0
        print(f'{name:<35} {latency:>9.0f}ms {tokens:>8} {tps:>10.0f} t/s')
        print(f'  Response: {content[:120]}')
        print()

await run_comparison()

[vLLM] (APIServer pid=7921) INFO:     127.0.0.1:60438 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Prompt: What are the three laws of thermodynamics? Be concise.

Provider                               Latency   Tokens        Speed
----------------------------------------------------------------------
Local A100 (Llama 3.1 8B)                3281ms      103         31 t/s
  Response: The three laws of thermodynamics are:

1. **Zeroth Law of Thermodynamics**: If two systems are in thermal equilibrium wi

Groq (Llama 3.3 70B)                      409ms       62        152 t/s
  Response: The three laws of thermodynamics are:

1. **First Law**: Energy cannot be created or destroyed, only converted.
2. **Sec

Mistral Small                            1132ms      114        101 t/s
  Response: Here are the three laws of thermodynamics, concisely stated:

1. **Zeroth Law**: If two systems are each in thermal equi



In [ ]:
# Cell 9: Keep alive
# Run this cell to keep the server running.
# The tunnel URL above stays valid as long as this cell runs.

import time

print(f'Server running at: {public_url}')
print('Keeping alive. Interrupt kernel to stop.')
print()

start = time.time()
while True:
    elapsed = time.time() - start
    hours = int(elapsed // 3600)
    minutes = int((elapsed % 3600) // 60)
    print(f'\rUptime: {hours:02d}h {minutes:02d}m | URL: {public_url}', end='')
    time.sleep(30)

## Wiring into NeuralOps

Once the server is running, add `LOCAL_A100` to your router in `sdk/neuralops/router.py`:

```python
class Provider(str, Enum):
    GROQ       = "groq"
    MISTRAL    = "mistral"
    OPENROUTER = "openrouter"
    LOCAL_A100 = "local_a100"   # add this

PROVIDERS: list[ProviderConfig] = [
    ProviderConfig(
        name=Provider.LOCAL_A100,
        base_url="https://YOUR-NGROK-URL.ngrok-free.app/v1/chat/completions",
        api_key_env="LOCAL_API_KEY",
        model="llama-3.1-70b",
    ),
    # ... existing providers
]
```

Add to `.env`:
```
LOCAL_API_KEY=local
```

The router will now prefer your local A100 (fastest) and fall back to Groq/Mistral if the tunnel goes down.

Every call through the local model is traced by NeuralOps. You can compare its causal chains, costs, and quality scores against the cloud providers in the benchmark arena.